In [2]:
import os
import json
import time
import pickle
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
import optuna
from optuna.samplers import TPESampler

In [3]:
load_dotenv()

True

In [4]:
# definindo os caminhos
data_path = os.getenv('DATA_PATH')
models_path = os.getenv('MODELS_PATH')
results_path = os.getenv('RESULTS_PATH')

In [5]:
# detectando se há GPU disponível para acelerar o treinamento
import subprocess

def cuda_available():
    try:
        subprocess.check_output(["nvidia-smi"])
        return True
    except Exception:
        return False


DEVICE = "cuda" if cuda_available() else "cpu"
print(f"dispositivo: {DEVICE}")

dispositivo: cpu


In [6]:
# carregando o dataset de treino do modelo 1
df = pd.read_parquet(os.path.join(data_path, 'model1_dataset.parquet'))
df.head()

,player_id,date,market_value_in_eur,height_in_cm,international_caps,international_goals,national_team_ranking_inv,position_rank,sub_pos_Attacking Midfield,sub_pos_Central Midfield,...,assists_per_90,minutes_per_game,squad_size,net_transfer_record,national_team_players,stadium_seats,league_tier,club_computed_market_value,year,log_market_value
0,6893,2003-12-15,900000,188.0,0.0,0.0,0,2,False,False,...,0.0,0.0,25,-1350000.0,4,26850,1,900000,2003,13.710151
1,14007,2004-10-04,750000,177.0,0.0,0.0,0,3,False,False,...,0.0,0.0,26,900000.0,8,18360,1,67650000,2004,13.527830
2,13957,2004-10-04,2000000,182.0,0.0,0.0,0,1,False,False,...,0.0,0.0,0,-0.0,0,5441,4,2900000,2004,14.508658
3,13952,2004-10-04,2000000,175.0,0.0,0.0,0,4,False,False,...,0.0,0.0,27,-24490000.0,12,50033,4,18050000,2004,14.508658
4,13894,2004-10-04,500000,193.0,0.0,0.0,0,1,False,False,...,0.0,0.0,26,-0.0,2,8432,4,4100000,2004,13.122365


In [7]:
# definindo as colunas de feature
NON_FEATURES = ["player_id", "date", "market_value_in_eur", "log_market_value"]
FEATURES = [c for c in df.columns if c not in NON_FEATURES]
FEATURES

['height_in_cm',
 'international_caps',
 'international_goals',
 'national_team_ranking_inv',
 'position_rank',
 'sub_pos_Attacking Midfield',
 'sub_pos_Central Midfield',
 'sub_pos_Centre-Back',
 'sub_pos_Centre-Forward',
 'sub_pos_Defensive Midfield',
 'sub_pos_Goalkeeper',
 'sub_pos_Left Midfield',
 'sub_pos_Left Winger',
 'sub_pos_Left-Back',
 'sub_pos_Right Midfield',
 'sub_pos_Right Winger',
 'sub_pos_Right-Back',
 'sub_pos_Second Striker',
 'foot_both',
 'foot_left',
 'foot_right',
 'confederation_encoded',
 'age',
 'age_squared',
 'goals_season',
 'assists_season',
 'minutes_season',
 'games_season',
 'yellow_cards_season',
 'red_cards_season',
 'goals_per_90',
 'assists_per_90',
 'minutes_per_game',
 'squad_size',
 'net_transfer_record',
 'national_team_players',
 'stadium_seats',
 'league_tier',
 'club_computed_market_value',
 'year']

In [8]:
# split temporal
df_train = df[df["date"] < "2022-01-01"].copy()
df_val = df[(df["date"] >= "2022-01-01") & (df["date"] < "2024-01-01")].copy()
df_test = df[df["date"] >= "2024-01-01"].copy()

X_train, y_train = df_train[FEATURES].astype(float), df_train["log_market_value"]
X_val, y_val = df_val[FEATURES].astype(float), df_val["log_market_value"]
X_test, y_test = df_test[FEATURES].astype(float), df_test["log_market_value"]

In [9]:
# hiperparâmetros iniciais
PARAMS_INIT = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",
    device=DEVICE,
    n_jobs=-1,
)

In [10]:
# treinando com os hiperparâmetros iniciais
model_init = XGBRegressor(**PARAMS_INIT)
model_init.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

pred_init = model_init.predict(X_val)
mape_init = (
    np.mean(np.abs((np.expm1(y_val) - np.expm1(pred_init)) / (np.expm1(y_val) + 1e-6)))
    * 100
)
r2_init = r2_score(y_val, pred_init)

print(f"MAPE (val, params iniciais): {mape_init:.2f} %")
print(f"R² (val, params iniciais): {r2_init:.4f}")

MAPE (val, params iniciais): 48.05 %
R² (val, params iniciais): 0.7598


In [11]:
# dividindo em 5 partes mantendo a ordem cronológica
tscv = TimeSeriesSplit(n_splits=5)


def objective(trial):
    # definindo os hiperparâmetros a serem testados
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 300, 1200),
        max_depth=trial.suggest_int("max_depth", 4, 7),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        subsample=trial.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 5, 20),
        gamma=trial.suggest_float("gamma", 0.0, 2.0),
        random_state=42,
        tree_method="hist",
        device=DEVICE,
        n_jobs=-1,
    )

    # guardando rmse das partes
    rmse_folds = []

    # treinando o modelo verificando as métricas
    for tr_idx, cv_idx in tscv.split(X_train):
        X_tr, X_cv = X_train.iloc[tr_idx], X_train.iloc[cv_idx]
        y_tr, y_cv = y_train.iloc[tr_idx], y_train.iloc[cv_idx]

        m = XGBRegressor(**params, early_stopping_rounds=30)
        m.fit(X_tr, y_tr, eval_set=[(X_cv, y_cv)], verbose=False)
        rmse_folds.append(np.sqrt(np.mean((y_cv.values - m.predict(X_cv)) ** 2)))

    return float(np.mean(rmse_folds))

In [12]:
# busca de hiperparâmetros com optuna
study = optuna.create_study(
    direction="minimize", sampler=TPESampler(seed=42), study_name="model1_market_value"
)
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"melhor RMSE (cv): {study.best_value:.5f}")
print(f"melhores params : {study.best_params}")

[I 2026-07-18 15:49:16,503] A new study created in memory with name: model1_market_value


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-07-18 15:50:19,744] Trial 0 finished with value: 0.6128001309797177 and parameters: {'n_estimators': 637, 'max_depth': 7, 'learning_rate': 0.07259248719561363, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 0.029375384576328288, 'reg_lambda': 0.13066739238053282, 'min_child_weight': 18, 'gamma': 1.2022300234864176}. Best is trial 0 with value: 0.6128001309797177.
[I 2026-07-18 15:50:48,046] Trial 1 finished with value: 0.6291983676988606 and parameters: {'n_estimators': 937, 'max_depth': 4, 'learning_rate': 0.13826189316223852, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.035113563139704075, 'reg_lambda': 0.2327067708383781, 'min_child_weight': 9, 'gamma': 1.0495128632644757}. Best is trial 0 with value: 0.6128001309797177.
[I 2026-07-18 15:51:35,397] Trial 2 finished with value: 0.6141379831202424 and parameters: {'n_estimators': 689, 'max_depth': 5, 'learning_rate': 0.05243180891902853, 'subsamp

In [13]:
# top-10 combinações testadas pelo optuna e seus rmse médios
df_trials = study.trials_dataframe(attrs=("number", "value", "params"))
df_top10 = df_trials.sort_values("value").head(10).reset_index(drop=True)

param_cols = [c for c in df_top10.columns if c.startswith("params_")]
df_top10[["number", "value"] + param_cols]

,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample
0,89,0.588041,0.762265,0.146379,0.046641,7,7,1086,0.063871,0.375443,0.742514
1,97,0.588195,0.708114,0.143323,0.057420,7,6,1084,0.064889,0.208120,0.773399
2,92,0.588402,0.728093,0.088955,0.046549,7,6,1042,0.048898,0.390186,0.744023
3,95,0.588501,0.740248,0.200518,0.045843,7,5,1081,0.143882,0.268329,0.753603
4,91,0.588761,0.731532,0.150710,0.046565,7,7,1085,0.049025,0.368941,0.743934
5,82,0.588873,0.738008,0.094580,0.042566,7,9,1126,0.087532,0.725168,0.758497
6,72,0.588998,0.750242,0.144142,0.049726,7,10,1091,0.050208,0.824577,0.726177
7,74,0.589439,0.764795,0.204882,0.044942,7,10,1077,0.075207,0.427277,0.755962
8,68,0.589637,0.658997,0.105501,0.050769,7,12,1096,0.074824,0.647623,0.724186
9,77,0.589733,0.788842,0.159727,0.057801,7,10,1017,0.052832,0.728207,0.755709


In [14]:
# treinando o modelo final com os melhores hiperparâmetros encontrados pelo optuna
best_params = {
    **study.best_params,
    "random_state": 42,
    "tree_method": "hist",
    "device": DEVICE,
    "n_jobs": -1,
}
model_final = XGBRegressor(**best_params)
model_final.fit(X_train, y_train, verbose=100)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7622645837959666, device='cpu',
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.14637907535670713,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.04664102904532205,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=7, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1086, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [15]:
# métricas do modelo final na validação
pred_final = model_final.predict(X_val)

y_val_real = np.expm1(y_val.values)
pred_real = np.expm1(pred_final)

mae_final = np.mean(np.abs(y_val_real - pred_real))
mape_final = np.mean(np.abs((y_val_real - pred_real) / (y_val_real + 1e-6))) * 100
rmse_final = np.sqrt(np.mean((y_val_real - pred_real) ** 2))
rmse_log = np.sqrt(np.mean((y_val.values - pred_final) ** 2))
r2_final = r2_score(y_val, pred_final)

print(f"MAE: € {mae_final:>12,.0f}")
print(f"MAPE: {mape_final:>12.2f} % (alvo < 20%)")
print(f"RMSE (€): € {rmse_final:>12,.0f}")
print(f"RMSE (log): {rmse_log:>12.4f} (alvo < 0.5)")
print(f"R² (log): {r2_final:>12.4f} (alvo >= 0.80)")

MAE: €    1,640,144
MAPE:        45.94 % (alvo < 20%)
RMSE (€): €    4,052,766
RMSE (log):       0.5425 (alvo < 0.5)
R² (log):       0.7730 (alvo >= 0.80)


In [16]:
# comparando com o baseline para confirmar melhoria
with open(os.path.join(results_path, "baseline_model1_metrics.json")) as f:
    baseline = json.load(f)

melhoria = (baseline["MAPE_pct"] - mape_final) / baseline["MAPE_pct"] * 100
print(f"MAPE baseline: {baseline['MAPE_pct']:.2f} %")
print(f"MAPE modelo: {mape_final:.2f} %")
print(f"melhoria: {melhoria:.1f} %")

MAPE baseline: 70.13 %
MAPE modelo: 45.94 %
melhoria: 34.5 %


In [17]:
# salvando o modelo final e a lista de features
model_final.save_model(os.path.join(models_path, "model1_market_value.json"))


with open(os.path.join(models_path, "feature_names_model1.pkl"), "wb") as f:
    pickle.dump(FEATURES, f)

In [18]:
# salvando as métricas de validação
metrics = {
    "dataset": "validation",
    "split": {
        "train_end": "2021-12-31",
        "val_start": "2022-01-01",
        "val_end": "2023-12-31",
    },
    "n_train": int(len(X_train)),
    "n_val": int(len(X_val)),
    "n_features": int(len(FEATURES)),
    "device": DEVICE,
    "MAE_eur": float(mae_final),
    "MAPE_pct": float(mape_final),
    "RMSE_eur": float(rmse_final),
    "RMSE_log": float(rmse_log),
    "R2_log": float(r2_final),
    "best_params": study.best_params,
    "optuna_trials": 100,
    "optuna_cv": "TimeSeriesSplit(n_splits=5)",
    "optuna_best_rmse": float(study.best_value),
    "baseline_mape_pct": float(baseline["MAPE_pct"]),
    "improvement_pct": float(melhoria),
}

out_path = os.path.join(results_path, "metrics_model1_val.json")
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)